# Multimodal Emotion Recognition — Feature Fusion (Option 1)

Combines **openSMILE IS10 audio features** (1582 dims) with **RoBERTa CLS embeddings** (768 dims) extracted from the fine-tuned text model, then trains a 2-layer MLP on the concatenated 2350-dim vector.

| Input | Source | Dims |
|---|---|---|
| Audio (IS10) | `extract_audio_features.py` | 1582 |
| Text CLS | Fine-tuned RoBERTa (`text_experiments.ipynb`) | 768 |
| **Fused** | Concatenation | **2350** |

IS10 is used because `audio_lr_is10` is the best unimodal audio model. Run cells **top-to-bottom**. CLS embeddings are cached to `data/features/roberta_cls_trial{N}/` after the first extraction — re-running skips the expensive forward pass.

In [6]:
import sys, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset as TorchDataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, RobertaTokenizer

SRC = Path('../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import (
    FEATURES_DIR, CHECKPOINTS_DIR, METRICS_DIR, PREDICTIONS_DIR, OUTPUTS_DIR,
    EMOTION_LABELS, NUM_CLASSES, RANDOM_SEED, FEATURE_SETS,
    audio_tag, multimodal_tag, get_model_path, get_scaler_path,
    get_hparams_path, get_metrics_path, get_prediction_path,
)
from evaluate import evaluate_and_save

for d in (CHECKPOINTS_DIR, METRICS_DIR, PREDICTIONS_DIR, OUTPUTS_DIR / 'plots'):
    d.mkdir(parents=True, exist_ok=True)

# ── Configuration ────────────────────────────────────────────────────────────
AUDIO_FEATURE_SET  = 'is10'    # 1582 dims — best unimodal audio model
BEST_TRIAL         = '5'       # Optuna trial from text_experiments.ipynb
ROBERTA_MODELS_DIR = Path('../data/models')
TAG = multimodal_tag('early_fusion')  # -> 'multimodal_late_fusion'

# Best hyperparameters from text Optuna search (Trial 5)
ROBERTA_BEST_PARAMS = {
    'learning_rate':               8.195663562822106e-06,
    'per_device_train_batch_size': 8,
    'num_train_epochs':            4,
    'weight_decay':                0.07960545199184652,
}

torch.manual_seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device         : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
print(f'Tag            : {TAG}')
print(f'Audio features : {AUDIO_FEATURE_SET}  ({FEATURE_SETS[AUDIO_FEATURE_SET][0]}, 1582 dims)')

Device         : cpu
Tag            : multimodal_early_fusion
Audio features : is10  (IS10, 1582 dims)


## 1 · Load & Align Data

Reads the manifest CSVs (built by `build_manifest.py`) for each split, constructs the **context-linked text** column (same logic as `text_experiments.ipynb`: prepend previous utterance with `</s>` separator), then inner-joins with the IS10 audio feature parquets on `(dialogue_id, utterance_id)`.

In [7]:
MANIFEST_DIR = Path('../data/processed')

def build_context(row):
    prev = row['prev_text']
    return (prev + ' </s> ' + row['text']) if prev != '' else row['text']

def load_manifest(split):
    df = pd.read_csv(MANIFEST_DIR / f'manifest_{split}.csv')
    df['prev_text'] = df.groupby('dialogue_id')['text'].shift(1).fillna('')
    df['text_with_context'] = df.apply(build_context, axis=1)
    return df[['dialogue_id', 'utterance_id', 'text_with_context']]

def load_aligned(split):
    manifest = load_manifest(split)
    audio_df = pd.read_parquet(FEATURES_DIR / f'{split}_{AUDIO_FEATURE_SET}.parquet')
    audio_df = audio_df.dropna().reset_index(drop=True)
    merged   = audio_df.merge(manifest, on=['dialogue_id', 'utterance_id'], how='inner')
    missed   = len(audio_df) - len(merged)
    if missed:
        print(f'[WARN] {split}: {missed} audio rows had no text match (dropped)')
    print(f'{split:5s}: {len(merged):,} aligned utterances')
    return merged

train_df = load_aligned('train')
dev_df   = load_aligned('dev')
test_df  = load_aligned('test')

ID_COLS    = {'dialogue_id', 'utterance_id', 'label_idx', 'text_with_context'}
AUDIO_COLS = [c for c in train_df.columns if c not in ID_COLS]
print(f'\nAudio feature dims : {len(AUDIO_COLS)}')
print(f'Sample (train[0])  : {train_df["text_with_context"].iloc[0][:80]!r}')

train: 9,982 aligned utterances
dev  : 1,106 aligned utterances
test : 2,610 aligned utterances

Audio feature dims : 1582
Sample (train[0])  : "also I was the point person on my company's transition from the KL-5 to GR-6 sys"


### 1b · Ensure Audio Baseline (LR + IS10)

Checks whether `audio_lr_is10` has already been trained. If not, runs it using the pre-saved best hyperparameters from `config.py` (`--no-optimize`) — takes ~30 s on CPU.

In [8]:
from audio.lr.train import main as _train_audio_lr

_audio_is10_metrics = get_metrics_path(audio_tag('lr', 'is10'))

if _audio_is10_metrics.exists():
    _m = json.loads(_audio_is10_metrics.read_text())
    print('Audio LR IS10 already trained:')
    print(f'  Test macro-F1    : {_m["macro_f1"]:.4f}')
    print(f'  Test weighted-F1 : {_m["weighted_f1"]:.4f}')
else:
    print('Audio LR IS10 not found — training now (saved best params, no Optuna search)...')
    _train_audio_lr('is10', optimize=False)
    print('Audio LR IS10 training complete.')

Audio LR IS10 already trained:
  Test macro-F1    : 0.1743
  Test weighted-F1 : 0.2493


## 2 · RoBERTa Checkpoint

Looks for the fine-tuned checkpoint at `data/models/optuna_trial_{BEST_TRIAL}/`. If found, CLS embeddings are extracted directly. If **not found** (e.g. model was trained on a cluster and not downloaded yet), the next cell re-trains RoBERTa locally using the fixed best hyperparameters from Trial 5.

In [9]:
def find_checkpoint(trial_dir):
    if not trial_dir.exists():
        return None
    checkpoints = sorted(
        [p for p in trial_dir.iterdir() if p.is_dir() and 'checkpoint' in p.name],
        key=lambda p: int(p.name.split('-')[-1]),
    )
    if checkpoints:
        return checkpoints[-1]
    best_model = trial_dir / 'best_model'
    return best_model if best_model.exists() else None

trial_dir = ROBERTA_MODELS_DIR / f'optuna_trial_{BEST_TRIAL}'
ckpt_path = find_checkpoint(trial_dir)

if ckpt_path:
    print(f'Checkpoint found : {ckpt_path}')
    ROBERTA_SOURCE = str(ckpt_path)
else:
    print(f'No checkpoint at : {trial_dir}')
    print('Run the next cell to train RoBERTa with the best hyperparameters.')
    ROBERTA_SOURCE = None

No checkpoint at : ../data/models/optuna_trial_5
Run the next cell to train RoBERTa with the best hyperparameters.


### 2b · Train RoBERTa (only if checkpoint is missing)

Uses the fixed best hyperparameters from Trial 5. **Skip this cell** if a checkpoint was found above.

In [ ]:
if ROBERTA_SOURCE is not None:
    print('Checkpoint already available — skipping training.')
else:
    from sklearn.metrics import accuracy_score
    from transformers import get_linear_schedule_with_warmup

    print('Training RoBERTa with best hyperparameters...')
    print(f'  learning_rate               : {ROBERTA_BEST_PARAMS["learning_rate"]:.2e}')
    print(f'  per_device_train_batch_size : {ROBERTA_BEST_PARAMS["per_device_train_batch_size"]}')
    print(f'  num_train_epochs            : {ROBERTA_BEST_PARAMS["num_train_epochs"]}')
    print(f'  weight_decay                : 0.01  (hardcoded in original objective)')
    print()

    roberta_save_dir = ROBERTA_MODELS_DIR / f'optuna_trial_{BEST_TRIAL}'
    roberta_save_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = roberta_save_dir / 'best_model'

    # ── Identical to text_experiments.ipynb ──────────────────────────────────
    tok_train = RobertaTokenizer.from_pretrained('roberta-base')

    class MELDTextDataset(TorchDataset):
        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.texts     = texts.tolist()
            self.labels    = labels.tolist()
            self.tokenizer = tokenizer
            self.max_len   = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text  = str(self.texts[idx])
            label = self.labels[idx]
            encoding = self.tokenizer(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            return {
                'input_ids':      encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels':         torch.tensor(label, dtype=torch.long),
            }

    def compute_metrics(preds, labels):
        return {
            'macro_f1':    f1_score(labels, preds, average='macro',    zero_division=0),
            'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
            'accuracy':    accuracy_score(labels, preds),
        }

    # Class weights — identical to text_experiments.ipynb
    _train_labels  = train_df['label_idx'].values
    class_weights  = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(_train_labels),
        y=_train_labels,
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    rob_train = AutoModelForSequenceClassification.from_pretrained(
        'roberta-base', num_labels=NUM_CLASSES
    ).to(DEVICE)

    TRAIN_BS = int(ROBERTA_BEST_PARAMS['per_device_train_batch_size'])
    N_EPOCHS  = int(ROBERTA_BEST_PARAMS['num_train_epochs'])

    tr_ld = DataLoader(
        MELDTextDataset(train_df['text_with_context'].values,
                        train_df['label_idx'].values, tok_train),
        batch_size=TRAIN_BS, shuffle=True,
    )
    dv_ld = DataLoader(
        MELDTextDataset(dev_df['text_with_context'].values,
                        dev_df['label_idx'].values, tok_train),
        batch_size=32, shuffle=False,
    )

    total_steps = len(tr_ld) * N_EPOCHS

    # weight_decay=0.01: hardcoded in the original objective (the Optuna-suggested
    # value was captured by study.best_params but never passed to TrainingArguments)
    optimizer = AdamW(
        rob_train.parameters(),
        lr           = ROBERTA_BEST_PARAMS['learning_rate'],
        weight_decay = 0.01,
    )
    # Linear LR decay to 0 — default scheduler in HF TrainingArguments
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )

    print(f'Train: {len(train_df):,} utterances  |  {len(tr_ld)} batches/epoch  |  {N_EPOCHS} epochs')
    print(f'Total steps: {total_steps}  |  LR decays linearly to 0\n')

    best_mf1, best_state = 0.0, None

    for epoch in range(1, N_EPOCHS + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        rob_train.train()
        pbar = tqdm(tr_ld, desc=f'Epoch {epoch}/{N_EPOCHS}', unit='batch')
        for batch in pbar:
            # Mirrors WeightedTrainer.compute_loss from text_experiments.ipynb
            labels = batch.pop('labels').to(DEVICE)
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = rob_train(**batch).logits
            loss   = nn.CrossEntropyLoss(weight=weights_tensor)(
                logits.view(-1, NUM_CLASSES), labels.view(-1)
            )
            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping — default max_grad_norm=1.0 in TrainingArguments
            torch.nn.utils.clip_grad_norm_(rob_train.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            pbar.set_postfix(loss=f'{loss.item():.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}')

        # ── Eval ──────────────────────────────────────────────────────────────
        rob_train.eval()
        preds_e, labels_e = [], []
        with torch.no_grad():
            for batch in tqdm(dv_ld, desc='  eval', leave=False, unit='batch'):
                labels_b = batch.pop('labels')
                batch    = {k: v.to(DEVICE) for k, v in batch.items()}
                logits   = rob_train(**batch).logits
                preds_e.extend(logits.argmax(1).cpu().tolist())
                labels_e.extend(labels_b.tolist())

        m    = compute_metrics(preds_e, labels_e)
        flag = '  ← best' if m['macro_f1'] > best_mf1 else ''
        print(f'Epoch {epoch}/{N_EPOCHS}  '
              f'macro_f1={m["macro_f1"]:.4f}  '
              f'weighted_f1={m["weighted_f1"]:.4f}  '
              f'accuracy={m["accuracy"]:.4f}{flag}')

        if m['macro_f1'] > best_mf1:
            best_mf1  = m['macro_f1']
            best_state = {k: v.cpu().clone() for k, v in rob_train.state_dict().items()}

    rob_train.load_state_dict(best_state)
    rob_train.save_pretrained(str(best_ckpt))
    tok_train.save_pretrained(str(best_ckpt))
    print(f'\nBest dev macro-F1 : {best_mf1:.4f}')
    print(f'Model saved       → {best_ckpt}')

    ckpt_path      = best_ckpt
    ROBERTA_SOURCE = str(best_ckpt)

    del rob_train
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

Training RoBERTa with best hyperparameters...
  learning_rate               : 8.20e-06
  per_device_train_batch_size : 8
  num_train_epochs            : 4
  weight_decay                : 0.01  (hardcoded in original objective)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train: 9,982 utterances  |  1248 batches/epoch  |  4 epochs
Total steps: 4992  |  LR decays linearly to 0



Epoch 1/4:   0%|          | 0/1248 [00:00<?, ?batch/s]

### 2c · Extract & Cache CLS Embeddings

Passes each utterance through `model.roberta` and saves the `[CLS]` token (index 0 of `last_hidden_state`, **768 dims**) per split. Cached to `data/features/roberta_cls_trial{N}/` — re-running is a no-op.

In [ ]:
CLS_CACHE_DIR = FEATURES_DIR / f'roberta_cls_trial{BEST_TRIAL}'
CLS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Show what is already cached vs what still needs extraction
split_dfs = [('train', train_df), ('dev', dev_df), ('test', test_df)]
print('CLS cache status:')
needs_extraction = []
for s, df in split_dfs:
    cached = (CLS_CACHE_DIR / f'{s}.npy').exists()
    status = 'cached' if cached else f'needs extraction  ({len(df):,} utterances)'
    print(f'  {s:5s}: {status}')
    if not cached:
        needs_extraction.append(s)

if needs_extraction:
    print(f'\nExtracting {len(needs_extraction)} split(s) from: {ROBERTA_SOURCE}')
    tok       = RobertaTokenizer.from_pretrained(ROBERTA_SOURCE)
    rob_model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_SOURCE, num_labels=NUM_CLASSES
    ).to(DEVICE)
    rob_model.eval()

    for split, df in split_dfs:
        cache = CLS_CACHE_DIR / f'{split}.npy'
        if cache.exists():
            continue
        texts     = df['text_with_context'].tolist()
        n_batches = (len(texts) + 31) // 32
        all_embs  = []
        for i in tqdm(range(0, len(texts), 32), total=n_batches,
                      desc=f'  {split}', unit='batch'):
            batch = texts[i : i + 32]
            enc = tok(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.no_grad():
                out = rob_model.roberta(**enc)
            all_embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        embs = np.vstack(all_embs).astype(np.float32)
        np.save(cache, embs)
        print(f'  {split}: saved  shape {embs.shape}')

    del rob_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
else:
    print('\nAll splits cached — skipping extraction.')

cls_train = np.load(CLS_CACHE_DIR / 'train.npy')
cls_dev   = np.load(CLS_CACHE_DIR / 'dev.npy')
cls_test  = np.load(CLS_CACHE_DIR / 'test.npy')
TEXT_DIM  = cls_train.shape[1]
print(f'\nCLS dims : train {cls_train.shape},  dev {cls_dev.shape},  test {cls_test.shape}')

## 3 · Build Fused Feature Vectors

Audio features are normalized with `StandardScaler` (fit on **train only** — no leakage). RoBERTa CLS embeddings come out of the model's LayerNorm already well-scaled, so they are concatenated directly.

```
audio (1582d, scaled) ‖ roberta_cls (768d)  →  2350-dim fused vector
```

In [ ]:
y_train = train_df['label_idx'].values.astype(int)
y_dev   = dev_df['label_idx'].values.astype(int)
y_test  = test_df['label_idx'].values.astype(int)

X_audio_train = train_df[AUDIO_COLS].values.astype(np.float32)
X_audio_dev   = dev_df[AUDIO_COLS].values.astype(np.float32)
X_audio_test  = test_df[AUDIO_COLS].values.astype(np.float32)

scaler = StandardScaler()
X_audio_train = scaler.fit_transform(X_audio_train)
X_audio_dev   = scaler.transform(X_audio_dev)
X_audio_test  = scaler.transform(X_audio_test)

scaler_file = get_scaler_path(TAG)
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Scaler saved → {scaler_file.name}')

X_train = np.concatenate([X_audio_train, cls_train], axis=1)
X_dev   = np.concatenate([X_audio_dev,   cls_dev],   axis=1)
X_test  = np.concatenate([X_audio_test,  cls_test],  axis=1)

INPUT_DIM = X_train.shape[1]
print(f'\nFused input dim : {len(AUDIO_COLS)} (audio) + {TEXT_DIM} (CLS) = {INPUT_DIM}')
print(f'Shapes          : train {X_train.shape}  dev {X_dev.shape}  test {X_test.shape}')

## 4 · Fusion MLP

Same architecture as `AudioMLP` in `src/audio/mlp/model.py`, scaled for the larger 2350-dim fused input. Uses `FocalLoss` with balanced class weights to handle MELD's heavy class imbalance (neutral dominates at ~47 %).

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        w     = self.weight.to(logits.device) if self.weight is not None else None
        ce    = F.nll_loss(log_p, targets, weight=w, reduction='none')
        p_t   = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()


class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout):
        super().__init__()
        layers, dim = [], input_dim
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            dim = h
        layers.append(nn.Linear(dim, NUM_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 5 · Optuna Hyperparameter Search

20-trial Bayesian search (TPE) on **dev macro-F1**. Hidden layer options are sized for the 2350-dim fused input.

| Hyperparameter | Search space |
|---|---|
| `hidden_1` | categorical: 512, 768, 1024 |
| `hidden_2` | categorical: 128, 256, 512 |
| `dropout` | float: [0.2, 0.5] |
| `lr` | log-uniform: [1e-4, 1e-2] |
| `focal_gamma` | float: [1.0, 3.0] |

In [ ]:
BATCH_SIZE      = 64
N_TRIALS        = 20
SEARCH_EPOCHS   = 30
SEARCH_PATIENCE = 5
FINAL_EPOCHS    = 100
FINAL_PATIENCE  = 10


def make_loader(X, y, shuffle):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle)


def class_weight_tensor(y):
    w = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y)
    return torch.tensor(w, dtype=torch.float32)


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    preds, labels = [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss   = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(y.cpu().tolist())
    return f1_score(labels, preds, average='macro', zero_division=0)


def train_model(X_tr, y_tr, X_dv, y_dv,
                hidden_1, hidden_2, dropout, lr, focal_gamma,
                max_epochs, patience, checkpoint=None):
    cw    = class_weight_tensor(y_tr).to(DEVICE)
    model = FusionMLP(INPUT_DIM, [hidden_1, hidden_2], dropout).to(DEVICE)
    crit  = FocalLoss(weight=cw, gamma=focal_gamma)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=5
    )
    tr_ld, dv_ld = make_loader(X_tr, y_tr, True), make_loader(X_dv, y_dv, False)
    best, no_impr = 0.0, 0
    for _ in range(1, max_epochs + 1):
        run_epoch(model, tr_ld, crit, opt)
        dev_mf1 = run_epoch(model, dv_ld, crit)
        sched.step(dev_mf1)
        if dev_mf1 > best:
            best, no_impr = dev_mf1, 0
            if checkpoint:
                torch.save(model.state_dict(), checkpoint)
        else:
            no_impr += 1
            if no_impr >= patience:
                break
    return best


def objective(trial):
    return train_model(
        X_train, y_train, X_dev, y_dev,
        hidden_1    = trial.suggest_categorical('hidden_1',    [512, 768, 1024]),
        hidden_2    = trial.suggest_categorical('hidden_2',    [128, 256, 512]),
        dropout     = trial.suggest_float('dropout',     0.2, 0.5),
        lr          = trial.suggest_float('lr',          1e-4, 1e-2, log=True),
        focal_gamma = trial.suggest_float('focal_gamma', 1.0,  3.0),
        max_epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE,
    )


print(f'Optuna search — {N_TRIALS} trials, metric: dev macro-F1')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)

best_params = study.best_params
print(f'\nBest dev macro-F1 : {study.best_value:.4f}')
print(f'Best params       : {best_params}')

hparams_file = get_hparams_path(TAG)
with open(hparams_file, 'w') as f:
    json.dump({**best_params, 'dev_mf1': round(study.best_value, 4)}, f, indent=2)
print(f'Hparams saved  → {hparams_file.name}')

## 6 · Final Training

Retrain with the best hyperparameters, longer epoch budget (`max=100`, `patience=10`), and save the best checkpoint.

In [ ]:
checkpoint = get_model_path(TAG)

print(f'Final training (max {FINAL_EPOCHS} epochs, patience={FINAL_PATIENCE})...')
best_dev_mf1 = train_model(
    X_train, y_train, X_dev, y_dev,
    best_params['hidden_1'], best_params['hidden_2'],
    best_params['dropout'],  best_params['lr'], best_params['focal_gamma'],
    max_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE,
    checkpoint=checkpoint,
)
print(f'Best dev macro-F1 : {best_dev_mf1:.4f}')
print(f'Checkpoint        → {checkpoint.name}')

## 7 · Evaluate on Test Set

In [ ]:
model = FusionMLP(
    INPUT_DIM,
    [best_params['hidden_1'], best_params['hidden_2']],
    best_params['dropout'],
).to(DEVICE)
model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
model.eval()

test_loader = make_loader(X_test, y_test, shuffle=False)
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(DEVICE))
        all_probs.append(F.softmax(logits, dim=1).cpu())
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

evaluate_and_save(
    y_true=np.array(all_labels),
    y_pred=np.array(all_preds),
    model_name=TAG,
)

softmax   = torch.cat(all_probs).numpy().astype(np.float32)
pred_file = get_prediction_path(TAG, 'test')
np.save(pred_file, softmax)
print(f'\nSoftmax saved → {pred_file.name}  shape {softmax.shape}')

## 8 · Text Baseline on Test Set (optional)

Evaluates the fine-tuned RoBERTa **classification head** on the test set, saving metrics to `outputs/metrics/text_roberta.json` for a consistent 3-way comparison (audio / text / fusion) on the same split.

Skipped if `text_roberta.json` already exists.

In [ ]:
TEXT_TAG          = 'text_roberta'
text_metrics_path = get_metrics_path(TEXT_TAG)

if text_metrics_path.exists():
    print(f'Text metrics already exist at {text_metrics_path.name} — skipping.')
else:
    print('Evaluating fine-tuned text model on test set...')

    class TextDataset(TorchDataset):
        def __init__(self, texts, labels, tok, max_len=128):
            self.texts  = [str(t) for t in texts]
            self.labels = list(labels)
            self.tok    = tok
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, i):
            enc = self.tok(
                self.texts[i],
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            return {k: v.squeeze(0) for k, v in enc.items()}, torch.tensor(self.labels[i])

    tok_text   = RobertaTokenizer.from_pretrained(ROBERTA_SOURCE)
    text_model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_SOURCE, num_labels=NUM_CLASSES
    ).to(DEVICE)
    text_model.eval()

    text_ds     = TextDataset(test_df['text_with_context'].values, y_test, tok_text)
    text_loader = DataLoader(text_ds, batch_size=32, shuffle=False)

    t_preds, t_labels, t_probs = [], [], []
    with torch.no_grad():
        for batch, labels in text_loader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = text_model(**batch).logits
            t_probs.append(F.softmax(logits, dim=1).cpu())
            t_preds.extend(logits.argmax(1).cpu().tolist())
            t_labels.extend(labels.tolist())

    evaluate_and_save(np.array(t_labels), np.array(t_preds), model_name=TEXT_TAG)
    text_softmax = torch.cat(t_probs).numpy().astype(np.float32)
    np.save(get_prediction_path(TEXT_TAG, 'test'), text_softmax)
    del text_model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

## 9 · Model Comparison

In [ ]:
MODELS_TO_COMPARE = {
    'Audio LR (IS10)':     'audio_lr_is10',      # best audio model
    'Audio LR (eGeMAPS)':  'audio_lr_egemaps',
    'Audio MLP (eGeMAPS)': 'audio_mlp_egemaps',
    'Text RoBERTa':        'text_roberta',
    'Fusion MLP':          TAG,
}

rows = []
for label, tag in MODELS_TO_COMPARE.items():
    p = get_metrics_path(tag)
    if p.exists():
        m = json.loads(p.read_text())
        rows.append({
            'Model':       label,
            'Macro F1':    m['macro_f1'],
            'Weighted F1': m['weighted_f1'],
        })
    else:
        print(f'[skip] {tag} — metrics not found')

df_cmp = pd.DataFrame(rows).sort_values('Macro F1', ascending=False).reset_index(drop=True)
print(df_cmp.to_string(index=False))

FUSION_COLOR  = '#C44E52'
TEXT_COLOR    = '#2CA02C'
DEFAULT_COLOR = '#4C72B0'

def bar_color(name):
    if name == 'Fusion MLP':    return FUSION_COLOR
    if name == 'Text RoBERTa':  return TEXT_COLOR
    return DEFAULT_COLOR

colors = [bar_color(r) for r in df_cmp['Model']]

fig, ax = plt.subplots(figsize=(8, 0.7 * len(df_cmp) + 1.5))
bars = ax.barh(df_cmp['Model'], df_cmp['Macro F1'],
               color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, df_cmp['Macro F1']):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}', va='center', fontsize=10,
    )
ax.set_xlabel('Macro F1 (test set)')
ax.set_xlim(0, df_cmp['Macro F1'].max() * 1.18)
ax.set_title('Model Comparison — Macro F1 (test set)')
ax.invert_yaxis()
plt.tight_layout()

plot_path = OUTPUTS_DIR / 'plots' / 'comparison_fusion.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved → {plot_path}')